# S&P 500 European Options Pricing and Model Validation

## Introduction

## Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib as plt
import scipy as sp
import fractions as fr

## S&P 500 Historical Data

In [ ]:
df_index = yf.download('^GSPC', period='2y')
df_close = df_index['Close']
S = df_close.iloc[-1,0]
q = 0.0104 #https://www.dividenddata.co.uk/sp500-dividend-yields.py
print(f'The most recent closing price from this data is {S}')
print(f'{q} is the continuous dividend yield, which has been estimated from the most recently published S&P 500 dividend yield. For the model, assume this stays constant.')
df_close

## Estimating Volatility

In [ ]:
df_returns = np.log(df_close/df_close.shift(1)).dropna()
sigma = (df_returns.std()).iloc[0] * np.sqrt(252)
print(f'{sigma} is our estimate of the annualised volatility.')


## User inputs

In [ ]:
allowed_type = ['call', 'Call', 'Put', 'put']
while True: #user enters call or put
    type = input('If the option is a call option, enter call. If the option is a put option, enter put.')
    if type in allowed_type:
        break
    else:
        print('Please enter either call or put.')

while True: #user enters maturity
    try:
        T = float(fr.Fraction(input('Enter the maturity time of the option in years.')))
    except ValueError:
        print('Please enter a number')
        continue
    if T>=1/12 and T<=30:
        break
    else:
        print('Please enter a number of years between 1/12 and 30.')

while True: #user enters strike price
    try:
        k = float(input('Enter the strike price'))
    except ValueError:
        print('Please enter a number')
        continue
    if k>0:
        break
    else:
        print('Please enter a positive number.')

## Approximating r

In [ ]:
df_rates = (pd.read_csv('daily-treasury-rates.csv')).head(1) #US daily treasury rates are low risk so base estimation off this
maturities = np.array([1/12, 1/8, 1/6, 1/4, 1/3, 1/2, 1, 2, 3, 5, 7, 10, 20, 30]) #in years
rates = (df_rates.iloc[0,1:]).to_numpy(dtype=np.float64) / 100
r = np.interp(T, maturities, rates)
print(f'{r:.4} is the approximated annualised risk free interest rate')